Goal: implement forward propagation and backpropagation from scratch in numpy (no PyTorch/autograd), verified against the textbook's own worked example, then applied to a real small dataset.





Recreate the book's toy network exactly. Use the same architecture from the text's Example 11.1.3 / Figure 11.2: 3 inputs → hidden layer of 2 (ReLU) → hidden layer of 2 (ReLU) → 3 outputs → softmax, with the exact weight matrices given in the chapter. Run forward propagation with input x=(1,1,1) and confirm your code reproduces the book's own numbers exactly: h1=(0,2), h2=(4,0), o=(-4,8,4), softmax ≈ (0.00, 0.98, 0.02). This is your sanity check before backprop — if forward prop doesn't match, fix that first.





Implement backpropagation from scratch, following the four-step structure in section 11.4.2: output layer gradient via a one-hot vector, hidden layer Jacobians via the ReLU derivative mask (1(z>0) on the diagonal), weight gradients via outer products. Compute the gradients for the same toy network and input.
Move to a real dataset — reuse load_digits() (no new data-loading needed). Build a small FFNN, e.g. 64 → 32 → 10, ReLU hidden layer, softmax output, cross-entropy loss.





Implement vectorized forward pass,
Backward pass, same equations as step 2
Basic SGD training loop
Train/test split (you've done this three times now)
Track train/test loss and accuracy per epoch

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
print(digits.keys())
X = digits.data
Y = digits.target
for key in digits.keys():
  value = digits[key]
  if hasattr(value, 'shape'):
    print(key, value.shape)
  else:
    print(key, type(value), '(no shape attr)')
##solve the feedforward neural network, weights provided in chapter -> calculation 1: multiply first round of weights by input x1, apply ReLU, calculation 2: output of calculation one becomes input for calculation 2, multiply by second round of weights, apply ReLU, and so on until the end
def ReLU(W, h):
  z = np.maximum(W @ h, 0)
  return z
def forward_calculation(W, h):
    h = ReLU(W, h)
    return h
output1 = forward_calculation(W = np.array([[2, -3, 0], [-1, 1, 2]]), h = np.array([1, 1, 1]))
output2 = forward_calculation(W = np.array([[1, 2], [2, -2]]), h = output1)
#don't apply ReLU to the final output, instead need to apply Softmax to convert final values into probabilities
output3 = np.array([[-1, 2], [2, 1], [1, 2]]) @ output2
output_exp = np.exp(output3)
softmax = []
for output in output3:
  softmax.append(np.exp(output)/np.sum(output_exp))
print(softmax)
#backpropagation - have a loss, want to know how much each weight in the network contributes to that wrongness, works backwords from output
#step 1 - compute the gradient of the output layer(subtract each softmax value by its true label)
true_label = 0
delta3 = softmax.copy()
delta3[true_label] -= 1
#step 2 - compute jacobian with respect to preceeding layer (take downstream error, resized to current layer's dimensions via transpose, and multiply by current Jacobian, for ReLU derivative is 0 or 1 depending on if the input was positive or negative)
def pre_activation(W, h):
  z = W @ h
  mask = z>0
  return mask
mask2 = pre_activation(W = np.array([[1, 2], [2, -2]]), h = output1)
#delta2 = (W3.T @ delta3) * mask2
delta2 = (np.array([[-1, 2], [2, 1], [1, 2]]).T @ delta3) * mask2
mask1 = pre_activation(W = np.array([[2, -3, 0], [-1, 1, 2]]), h = np.array([1, 1, 1]))
delta1 = (np.array([[1, 2], [2, -2]]).T @ delta2) * mask1
#step 3 - calculate weight gradients(a weight's gradient depends on both how wrong the output was, as well as how active the input was that this weight was multiplied with)
weight_gradient_1 = np.outer(delta1, np.array([1, 1, 1]))
weight_gradient_2 = np.outer(delta2, output1)
weight_gradient_3 = np.outer(delta3, output2)
#FFNN using real data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)
X_train_scaled = X_train/16.0
X_test_scaled = X_test/16.0
#X_train = 1437x64, X_test = 360x64, Y_train = 1437x1, Y_test = 360x1
W1 = np.random.randn(32, 64)*0.01 #W1 = 32x64
W2 = np.random.randn(10, 32)*0.01 #W2 = 10x32
#forward pass
Output1 = forward_calculation(W = W1, h = X_train_scaled.T) #Output1 = 32x1437
Output2 = W2 @ Output1 #Output2 = 10x1437
Softmax = np.exp(Output2)/np.sum(np.exp(Output2), axis = 0, keepdims=True) ##Softmax = 10x1437
print(Softmax)
#apply gradient descent
def forward(W1, W2, h):
  Output1 = forward_calculation(W1, h)
  Output2 = W2 @ Output1
  Softmax = np.exp(Output2)/np.sum(np.exp(Output2), axis = 0, keepdims=True)
  return Output1, Output2, Softmax
def backprop(W1, W2, h, Y):
  Output1, Output2, Softmax = forward(W1, W2, h)
  one_hot = np.eye(10)[Y] #one_hot = 1437x10
  Delta2 = Softmax.copy() - one_hot.T #Delta2 = 10x1437
  Mask1 = pre_activation(W1, h) #Mask1 = 32x1437
  Delta1 = (W2.T @ Delta2) * Mask1 #Delta1 = (32x1437)*(32x1437) = 32x1437
  Weight_Gradient_1 = Delta1 @ h.T #want Weight_Gradient_1 to match W1's shape -> 32x64
  Weight_Gradient_2 = Delta2 @ Output1.T #Weight_Gradient_2 = 10x32
  return Weight_Gradient_1, Weight_Gradient_2
eta = 0.001
n_steps = 10000
h = X_train_scaled.T
for i in range(n_steps):
    Weight_Gradient_1, Weight_Gradient_2 = backprop(W1, W2, h, Y_train)
    W1 = W1 - (eta*Weight_Gradient_1)
    W2 = W2 - (eta*Weight_Gradient_2)
Output1_test, Output2_test, Softmax_test = forward(W1, W2, X_test_scaled.T);
test_prediction = np.argmax(Softmax_test, axis=0)
test_correct = np.array(test_prediction==Y_test)
test_accuracy = np.sum(test_correct)/len(Y_test)
Output1_train, Output2_train, Softmax_train = forward(W1, W2, X_train_scaled.T);
train_prediction = np.argmax(Softmax_train, axis=0)
train_correct = np.array(train_prediction==Y_train)
train_accuracy = np.sum(train_correct)/len(Y_train)
print(train_accuracy, test_accuracy)

dict_keys(['data', 'target', 'frame', 'feature_names', 'target_names', 'images', 'DESCR'])
data (1797, 64)
target (1797,)
frame <class 'NoneType'> (no shape attr)
feature_names <class 'list'> (no shape attr)
target_names (10,)
images (1797, 8, 8)
DESCR <class 'str'> (no shape attr)
[np.float64(6.033664854558336e-06), np.float64(0.9820078648958168), np.float64(0.01798610143932864)]
[[0.1000066  0.10005534 0.10003721 ... 0.1000096  0.09983415 0.09990414]
 [0.10008969 0.10005716 0.10000206 ... 0.10008549 0.10008486 0.10012867]
 [0.10006921 0.10009278 0.09989418 ... 0.10011206 0.09984303 0.09994561]
 ...
 [0.09997483 0.09980279 0.09985005 ... 0.09986231 0.09985846 0.09987862]
 [0.09998589 0.10001581 0.10010375 ... 0.10009816 0.10021481 0.10012881]
 [0.09993293 0.0998718  0.09981728 ... 0.09977136 0.09979695 0.09984288]]
0.9958246346555324 0.9361111111111111
